In [1]:
import shutil

import pandas as pd
import pyiron_workflow as pwf
from ase.calculators import emt
from pyiron_workflow_atomistics import engine as engine_mod

from demonstrators import elastic_nodes

In [2]:
engine = engine_mod.ASEEngine(
    EngineInput=None,
    calculator=emt.EMT(),
    working_directory="demo_runs",
)

# Elastic constants

In [3]:
elastic_wf = pwf.node(elastic_nodes.unary_elastic_tensor)

elastic_wf.validate(do_ontology=False)
# Ontology can't handle constants yet

Type validation for 'unary_elastic_tensor' (valid=True, complete=False):
	subreports:
	Type validation for 'unary_elastic_tensor.elastic_constants_0' (valid=True, complete=False):
		unfulfilled edges:
			with_calc_input_0.output_0->calculate_0.engine
			get_attr_0.attr->generate_mp_deformations_0.structure
			generate_mp_deformations_0.deformed_structures->evaluate_structures_0.structures
			with_calc_input_1.output_0->evaluate_structures_0.engine
			get_attr_0.attr->fit_elastic_tensor_0.structure
			get_attr_0.attr->elastic_constants_summary_0.structure
		subreports:
		constant_0: <NOT PARSEABLE>
	constant_0: <NOT PARSEABLE>
None

In [4]:
elastic_run = elastic_wf.run(
    engine=engine.with_working_directory("elastic"),
    symbol="Au",
    relaxation_config=engine_mod.CalcInputMinimize(relax_cell=True)
)

print("IEEE elastic tensor")
pd.options.display.float_format = '{:.1f}'.format
pd.DataFrame(
    elastic_run.outputs.tensor_ieee,
    index=[f"{i+1}" for i in range(6)],
    columns=[f"{j+1}" for j in range(6)],
)

      Step     Time          Energy          fmax
BFGS:    0 15:12:45        0.002606        0.308859
BFGS:    1 15:12:45        0.000032        0.077465
BFGS:    2 15:12:45       -0.000135        0.002603
IEEE elastic tensor


,1,2,3,4,5,6
1,196.8,162.8,162.8,-0.0,0.0,0.0
2,162.8,196.8,162.8,-0.0,0.0,0.0
3,162.8,162.8,196.8,-0.0,0.0,0.0
4,-0.0,-0.0,-0.0,54.9,0.0,0.0
5,0.0,0.0,0.0,0.0,54.9,-0.0
6,0.0,0.0,0.0,0.0,-0.0,54.9


## Grain boundary segregation

## Binary phase stability

## Cleanup

In [5]:
shutil.rmtree(engine.working_directory)

In [11]:
import pathlib

p = pathlib.Path(engine.working_directory)


False